# Train Path A + C in Colab

This notebook prepares a FEVER-trained verifier (A) and a verified synthetic repair editor (C). Run it top to bottom in a **GPU runtime**. It writes data, checkpoints, and provenance to your own Google Drive. The bounded pilot is the default; it is a functionality check, not a publishable project benchmark.

Dataset: [official FEVER files and format](https://fever.ai/dataset/fever.html). Workflow and interpretation: [training_workflow.md](https://github.com/harshitap1305/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers/blob/codex/train-verifier-repair/docs/training_workflow.md). Colab resources and GPU availability vary; see the [Colab FAQ](https://research.google.com/colaboratory/faq.html).

## 1. Configure, mount Drive, clone the branch, install dependencies

Set `PROJECT_HOLDOUT_MANIFEST` to a JSON file in Drive before a non-pilot run. It must contain lists named `article_ids`, `entities`, `prompt_ids`, and `prompts` drawn from the held-out course-project set. A non-pilot run requires at least one course-project article ID or entity because wiki-source candidates have no prompt fields. Leave `RUN_END_TO_END` false until gold files and a Groq secret exist. Reuse a `RUN_NAME` only with the same inputs and settings.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

PILOT = True
RUN_NAME = "pilot_v1" if PILOT else "full_v1"
REPO_URL = "https://github.com/harshitap1305/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers.git"
REPO_REF = "codex/train-verifier-repair"
PROJECT_HOLDOUT_MANIFEST = ""  # Example: /content/drive/MyDrive/course_gold/holdout.json
PRECOMPUTED_TRAIN_RETRIEVAL = ""  # Optional, useful for large FEVER runs
PRECOMPUTED_DEV_RETRIEVAL = ""
RUN_END_TO_END = False

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/hallucination_project_ac")
RUN_DIR = DRIVE_ROOT / "runs" / RUN_NAME
FEVER_DIR = DRIVE_ROOT / "fever"
REPO_DIR = Path("/content/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers")
for folder in (DRIVE_ROOT, RUN_DIR, FEVER_DIR):
    folder.mkdir(parents=True, exist_ok=True)
os.environ["LLM_CACHE_PATH"] = str(RUN_DIR / "llm_cache.db")
os.environ["RETRIEVAL_CACHE_PATH"] = str(RUN_DIR / "retrieval_cache.db")
if not PILOT:
    if not PROJECT_HOLDOUT_MANIFEST or not Path(PROJECT_HOLDOUT_MANIFEST).is_file():
        raise ValueError("A non-pilot run requires PROJECT_HOLDOUT_MANIFEST.")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", REPO_REF,
                    REPO_URL, str(REPO_DIR)], check=True)
else:
    branch = subprocess.run(["git", "-C", str(REPO_DIR), "branch", "--show-current"],
                            capture_output=True, text=True, check=True).stdout.strip()
    if branch != REPO_REF:
        raise ValueError(f"Existing clone is on {branch!r}, expected {REPO_REF!r}")
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("Git commit:", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(REPO_DIR / "requirements-training.txt")], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun setup.")
hardware = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout
(RUN_DIR / "hardware.txt").write_text(hardware, encoding="utf-8")
(RUN_DIR / "pip-freeze.txt").write_text(
    subprocess.run([sys.executable, "-m", "pip", "freeze"],
                   capture_output=True, text=True, check=True).stdout, encoding="utf-8")
print(hardware)

def run_script(*args):
    command = [sys.executable, *map(str, args)]
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=REPO_DIR, check=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## 2. Download the official FEVER files

The downloads live in Drive. A successful download is renamed from `.part`; rerunning resumes an interrupted `.part` file. The wiki archive is large and Drive needs sufficient free space.

In [ ]:
URLS = {
    "train.jsonl": "https://fever.ai/download/fever/train.jsonl",
    "shared_task_dev.jsonl": "https://fever.ai/download/fever/shared_task_dev.jsonl",
    "wiki-pages.zip": "https://fever.ai/download/fever/wiki-pages.zip",
}
def download_once(name, url):
    target = FEVER_DIR / name
    if not target.exists():
        partial = FEVER_DIR / (name + ".part")
        subprocess.run(["wget", "--continue", "--tries=5", "--timeout=60",
                        "-O", str(partial), url], check=True)
        if partial.stat().st_size == 0:
            raise RuntimeError(f"Empty download: {url}")
        partial.replace(target)
    return target

FEVER_TRAIN = download_once("train.jsonl", URLS["train.jsonl"])
FEVER_DEV = download_once("shared_task_dev.jsonl", URLS["shared_task_dev.jsonl"])
WIKI_ZIP = download_once("wiki-pages.zip", URLS["wiki-pages.zip"])
with zipfile.ZipFile(WIKI_ZIP) as archive:
    if not any(name.endswith(".jsonl") for name in archive.namelist()):
        raise ValueError("Downloaded wiki archive has no JSONL pages.")
for path in (FEVER_TRAIN, FEVER_DEV):
    with path.open(encoding="utf-8") as stream:
        json.loads(stream.readline())
provenance = {
    "git_commit": commit, "pilot": PILOT, "run_name": RUN_NAME,
    "seed": 42, "data_sha256": {p.name: sha256_file(p)
                               for p in (FEVER_TRAIN, FEVER_DEV, WIKI_ZIP)},
    "urls": URLS,
}
(RUN_DIR / "input_provenance.json").write_text(
    json.dumps(provenance, indent=2) + "\n", encoding="utf-8")
print("FEVER files and hashes recorded in", RUN_DIR / "input_provenance.json")

## 3. Reserve held-out articles and course-project entities

The pilot excludes article titles in official FEVER development evidence from training and synthetic repair sources. A non-pilot run also requires your course-project test holdout manifest. This list is created before any training data is built. The verifier's final FEVER subset is filtered again against its actual training/development article keys.

In [ ]:
fields = ("article_ids", "entities", "prompt_ids", "prompts")
project = {field: [] for field in fields}
if PROJECT_HOLDOUT_MANIFEST:
    supplied = json.loads(Path(PROJECT_HOLDOUT_MANIFEST).read_text(encoding="utf-8"))
    for field in fields:
        values = supplied.get(field, [])
        if not isinstance(values, list) or not all(isinstance(v, str) for v in values):
            raise ValueError(f"Project holdout {field} must be a list of strings.")
        project[field] = [value.strip() for value in values if value.strip()]
if not PILOT and not (project["article_ids"] or project["entities"]):
    raise ValueError("Non-pilot run requires a course-project article ID or entity key; prompts alone cannot exclude wiki sources.")

fever_dev_articles = set()
with FEVER_DEV.open(encoding="utf-8") as stream:
    for line in stream:
        if not line.strip():
            continue
        row = json.loads(line)
        for group in row.get("evidence") or []:
            for item in group:
                if isinstance(item, list) and len(item) >= 3 and isinstance(item[2], str):
                    fever_dev_articles.add(item[2])
if not fever_dev_articles:
    raise ValueError("No article titles found in labelled FEVER development data.")
holdout = {field: sorted(set(project[field])) for field in fields}
holdout["article_ids"] = sorted(set(holdout["article_ids"]) | fever_dev_articles)
HOLDOUT_FILE = RUN_DIR / "holdout_manifest.json"
HOLDOUT_FILE.write_text(json.dumps(holdout, indent=2) + "\n", encoding="utf-8")
EXCLUDE_FILE = RUN_DIR / "excluded_verifier_keys.txt"
EXCLUDE_FILE.write_text(
    "\n".join(sorted(set(holdout["article_ids"]) | set(holdout["entities"]))) + "\n",
    encoding="utf-8")
print("Reserved official FEVER dev articles:", len(fever_dev_articles))
print("Holdout manifest:", HOLDOUT_FILE)

## 4. Extract Wikipedia pages and build a shared BM25 index

The index builder reads `data/raw/fever/wiki-pages` inside the clone. The pilot caps the index at 100,000 passages; the larger setting caps at 1,500,000 and may require high RAM. Both models and all four end-to-end variants use the same index. Existing Drive indexes are copied back to local disk when a Colab session restarts.

In [ ]:
WIKI_DIR = REPO_DIR / "data" / "raw" / "fever" / "wiki-pages"
EXTRACT_MARKER = WIKI_DIR / ".extract_complete"
if not EXTRACT_MARKER.exists():
    WIKI_DIR.parent.mkdir(parents=True, exist_ok=True)
    # Overwrite partial extraction from an interrupted Colab session.
    subprocess.run(["unzip", "-q", "-o", str(WIKI_ZIP),
                    "-d", str(WIKI_DIR.parent)], check=True)
    if not list(WIKI_DIR.glob("wiki-*.jsonl")):
        raise FileNotFoundError(f"Wiki JSONL pages missing after extraction: {WIKI_DIR}")
    EXTRACT_MARKER.write_text(provenance["data_sha256"]["wiki-pages.zip"] + "\n", encoding="utf-8")
if EXTRACT_MARKER.read_text(encoding="utf-8").strip() != provenance["data_sha256"]["wiki-pages.zip"]:
    raise ValueError("Extracted Wikipedia pages came from a different archive; use a new clone.")

MAX_PASSAGES = 100_000 if PILOT else 1_500_000
LOCAL_INDEX_DIR = REPO_DIR / "indexes"
LOCAL_INDEX_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_INDEX = LOCAL_INDEX_DIR / "bm25_A.pkl"
DRIVE_INDEX = RUN_DIR / "index" / "bm25_A.pkl"
if DRIVE_INDEX.exists():
    shutil.copy2(DRIVE_INDEX, LOCAL_INDEX)
else:
    run_script("scripts/build_index.py", "--tier", "A", "--bm25_only",
               "--max_passages", MAX_PASSAGES, "--output_dir", LOCAL_INDEX_DIR)
    if not LOCAL_INDEX.is_file():
        raise RuntimeError("Index builder did not produce bm25_A.pkl.")
    DRIVE_INDEX.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(LOCAL_INDEX, DRIVE_INDEX)
provenance["max_index_passages"] = MAX_PASSAGES
provenance["index_sha256"] = sha256_file(LOCAL_INDEX)
(RUN_DIR / "input_provenance.json").write_text(
    json.dumps(provenance, indent=2) + "\n", encoding="utf-8")
print("Shared index:", LOCAL_INDEX, "bytes:", LOCAL_INDEX.stat().st_size)

## 5. Retrieve evidence without reading claim labels

This step queries the same BM25 index for FEVER train and official-dev claims. With the repository's straightforward BM25 implementation, a full FEVER run can take a long time. For large runs, provide completed label-independent JSONL retrieval files through the `PRECOMPUTED_..._RETRIEVAL` settings above. The preparation scripts verify IDs and schemas.

In [ ]:
MAX_TRAIN_CLAIMS = 2_000 if PILOT else None
MAX_DEV_CLAIMS = 2_000 if PILOT else None

def retrieval_for(claims_file, name, limit, override):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    output = RUN_DIR / name
    if not output.exists():
        partial = RUN_DIR / (name + ".partial")
        args = ["scripts/retrieve_fever_evidence.py",
                "--fever-train", claims_file, "--index-path", LOCAL_INDEX,
                "--top-k", 5, "--output", partial, "--overwrite"]
        if limit is not None:
            args.extend(["--max-claims", limit])
        run_script(*args)
        partial.replace(output)
    return output

RETRIEVED_TRAIN = retrieval_for(
    FEVER_TRAIN, "retrieved_train.jsonl", MAX_TRAIN_CLAIMS,
    PRECOMPUTED_TRAIN_RETRIEVAL)
RETRIEVED_DEV = retrieval_for(
    FEVER_DEV, "retrieved_official_dev.jsonl", MAX_DEV_CLAIMS,
    PRECOMPUTED_DEV_RETRIEVAL)
print("Retrieval files:", RETRIEVED_TRAIN, RETRIEVED_DEV)

## 6. Prepare, train, and evaluate Path A

The FEVER train file is split by connected article/entity groups. Its internal dev split selects the verifier checkpoint. Evaluation uses a separate, filtered subset of official FEVER labelled development claims. Read the manifests before reporting a score, especially the held-out count and class balance.

In [ ]:
VERIFIER_DATA = RUN_DIR / "verifier"
if not (VERIFIER_DATA / "manifest.json").exists():
    args = ["scripts/prepare_fever_verifier.py",
            "--fever-train", FEVER_TRAIN, "--wiki-pages", WIKI_ZIP,
            "--retrieved", RETRIEVED_TRAIN, "--output-dir", VERIFIER_DATA,
            "--exclude-keys", EXCLUDE_FILE, "--seed", 42, "--overwrite"]
    if MAX_TRAIN_CLAIMS is not None:
        args.extend(["--max-claims", MAX_TRAIN_CLAIMS])
    run_script(*args)
print((VERIFIER_DATA / "manifest.json").read_text(encoding="utf-8")[:4000])

VERIFIER_RUN = RUN_DIR / "verifier_run"
VERIFIER_CKPT = VERIFIER_RUN / "final"
if not (VERIFIER_CKPT / "config.json").exists():
    args = ["scripts/train_verifier.py",
            "--train-file", VERIFIER_DATA / "train.jsonl",
            "--dev-file", VERIFIER_DATA / "dev.jsonl",
            "--output-dir", VERIFIER_RUN,
            "--model-name", "microsoft/deberta-v3-base",
            "--epochs", 1 if PILOT else 3,
            "--train-batch-size", 4, "--eval-batch-size", 8,
            "--gradient-accumulation", 2, "--precision", "auto", "--seed", 42]
    if list(VERIFIER_RUN.glob("checkpoint-*")):
        args.append("--resume")
    run_script(*args)
assert (VERIFIER_CKPT / "config.json").is_file()
print("Selected Path A checkpoint:", VERIFIER_CKPT)

In [ ]:
VERIFIER_HOLDOUT = RUN_DIR / "verifier_holdout"
if not (VERIFIER_HOLDOUT / "manifest.json").exists():
    args = ["scripts/prepare_fever_holdout.py",
            "--fever-dev", FEVER_DEV, "--wiki-pages", WIKI_ZIP,
            "--retrieved", RETRIEVED_DEV,
            "--train-file", VERIFIER_DATA / "train.jsonl",
            "--dev-file", VERIFIER_DATA / "dev.jsonl",
            "--output-dir", VERIFIER_HOLDOUT, "--seed", 42, "--overwrite"]
    if MAX_DEV_CLAIMS is not None:
        args.extend(["--max-claims", MAX_DEV_CLAIMS])
    run_script(*args)
print((VERIFIER_HOLDOUT / "manifest.json").read_text(encoding="utf-8"))

VERIFIER_EVAL = RUN_DIR / "verifier_eval"
if not (VERIFIER_EVAL / "metrics.json").exists():
    run_script("scripts/eval_verifier.py",
               "--eval-file", VERIFIER_HOLDOUT / "eval.jsonl",
               "--checkpoint", VERIFIER_CKPT, "--output-dir", VERIFIER_EVAL,
               "--bootstrap", 1000, "--seed", 42, "--overwrite")
print("Held-out Path A metrics:", VERIFIER_EVAL / "metrics.json")

## 7. Generate, verify, and split Path C candidates

Candidate edits are not training examples until the NLI verifier checks both the original and corrupted sentence. The non-pilot setting scans up to 100,000 source sentences; raise this cap only when compute and storage allow. Its verdict cache is append-only so an interrupted Colab session can resume filtering. The resulting test split is synthetic and article-disjoint; it does not measure editing of independently retrieved LLM answers.

In [ ]:
MAX_REPAIR_SOURCES = 4_000 if PILOT else 100_000
REPAIR_CANDIDATES = RUN_DIR / "repair_candidates.jsonl"
if not REPAIR_CANDIDATES.exists():
    partial = RUN_DIR / "repair_candidates.jsonl.partial"
    args = ["scripts/build_repair_data.py", "candidates",
            "--wiki-pages-dir", WIKI_DIR, "--holdout-manifest", HOLDOUT_FILE,
            "--output", partial, "--seed", 42]
    if MAX_REPAIR_SOURCES is not None:
        args.extend(["--max-sources", MAX_REPAIR_SOURCES])
    run_script(*args)
    partial.replace(REPAIR_CANDIDATES)
print("Unverified candidate rows:", sum(1 for _ in REPAIR_CANDIDATES.open(encoding="utf-8")))

REPAIR_DATA = RUN_DIR / "repair"
REPAIR_VERDICTS = RUN_DIR / "repair_verdicts.jsonl"
if not (REPAIR_DATA / "repair_manifest.json").exists():
    run_script("scripts/build_repair_data.py", "filter",
               "--candidates", REPAIR_CANDIDATES, "--output-dir", REPAIR_DATA,
               "--nli-model", "cross-encoder/nli-deberta-v3-small",
               "--verdicts-out", REPAIR_VERDICTS,
               "--unchanged-fraction", 0.2, "--seed", 42)
manifest = json.loads((REPAIR_DATA / "repair_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))
for split in ("train", "dev", "test"):
    path = REPAIR_DATA / f"repair_{split}.jsonl"
    if not path.is_file() or path.stat().st_size == 0:
        raise ValueError(f"Path C {split} split is empty; expand sources and rerun in a new RUN_NAME.")

## 8. Train and evaluate Path C

The pilot uses Flan-T5-small and one epoch to check the workflow on limited GPU time. A non-pilot run uses Flan-T5-base for three epochs. Select the best checkpoint from internal dev metrics, then evaluate it once on the article-disjoint synthetic test split.

In [ ]:
REPAIR_RUN = RUN_DIR / "repair_run"
BEST_REPAIR = REPAIR_RUN / "best.json"
target_epochs = 1 if PILOT else 3
completed_epochs = sorted(
    int(path.name.rsplit("-", 1)[1]) for path in REPAIR_RUN.glob("checkpoint-epoch-*")
    if path.name.rsplit("-", 1)[1].isdigit() and (path / "config.json").is_file()
)
if not completed_epochs or completed_epochs[-1] < target_epochs:
    train_args = ["scripts/train_repair.py",
               "--train-data", REPAIR_DATA / "repair_train.jsonl",
               "--dev-data", REPAIR_DATA / "repair_dev.jsonl",
               "--output-dir", REPAIR_RUN,
               "--model-name", "google/flan-t5-small" if PILOT else "google/flan-t5-base",
               "--epochs", target_epochs, "--batch-size", 2,
               "--grad-accum", 4, "--max-source-length", 320,
               "--max-target-length", 96, "--precision", "auto",
               "--policy", "rewrite+hedge", "--seed", 42]
    if completed_epochs:
        train_args.extend(["--resume-from", REPAIR_RUN / f"checkpoint-epoch-{completed_epochs[-1]}"])
    run_script(*train_args)
REPAIR_CKPT = Path(json.loads(BEST_REPAIR.read_text(encoding="utf-8"))["checkpoint"])
assert (REPAIR_CKPT / "config.json").is_file()
print("Selected Path C checkpoint:", REPAIR_CKPT)

REPAIR_EVAL = RUN_DIR / "repair_eval"
REPAIR_METRICS = REPAIR_EVAL / "metrics.json"
if not REPAIR_METRICS.exists():
    run_script("scripts/eval_repair.py",
               "--data", REPAIR_DATA / "repair_test.jsonl",
               "--split", "test", "--checkpoint", REPAIR_CKPT,
               "--metrics-out", REPAIR_METRICS,
               "--predictions-out", REPAIR_EVAL / "predictions.jsonl",
               "--n-bootstrap", 1000, "--policy", "rewrite+hedge", "--seed", 42)
print("Synthetic Path C test metrics:", REPAIR_METRICS)

## 9. Optional course-project held-out A/C comparison

This cell does nothing until `RUN_END_TO_END = True` in the first cell and you provide course-project gold dev/test files plus nonempty splits in Drive. Store `GROQ_API_KEY` in **Colab Secrets**, with notebook access enabled. The key is read at runtime and is never written to a file. All four systems use the same held-out inputs and index. Do not use the synthetic repair-test or FEVER-dev numbers as the final project A+C result.

In [ ]:
if RUN_END_TO_END:
    from google.colab import userdata
    COURSE_DIR = DRIVE_ROOT / "course_gold"
    GOLD_DEV = COURSE_DIR / "dev.jsonl"
    GOLD_TEST = COURSE_DIR / "test.jsonl"
    SPLITS = COURSE_DIR / "splits.json"
    for required in (GOLD_DEV, GOLD_TEST, SPLITS):
        if not required.is_file():
            raise FileNotFoundError(f"Course-project gold input missing: {required}")
    key = userdata.get("GROQ_API_KEY")
    if not key:
        raise ValueError("Set GROQ_API_KEY in Colab Secrets.")
    os.environ["GROQ_API_KEY"] = key
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "spacy>=3.7,<4"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_sm"], check=True)
    (RUN_DIR / "pip-freeze.txt").write_text(
        subprocess.run([sys.executable, "-m", "pip", "freeze"],
                       capture_output=True, text=True, check=True).stdout, encoding="utf-8")
    run_script("scripts/compare_training_paths.py",
               "--inputs", GOLD_TEST, "--gold-dev", GOLD_DEV,
               "--gold-test", GOLD_TEST, "--splits", SPLITS,
               "--verifier-checkpoint", VERIFIER_CKPT,
               "--repair-checkpoint", REPAIR_CKPT,
               "--index", LOCAL_INDEX,
               "--output-dir", RUN_DIR / "ac_comparison",
               "--n-bootstrap", 1000, "--seed", 42)
    print("Paired A/C comparison:", RUN_DIR / "ac_comparison")
else:
    print("Course-project comparison skipped: set RUN_END_TO_END after gold data and a Groq secret are ready.")

## 10. Check what the run produced

Inspect `input_provenance.json`, the FEVER and repair manifests, selected checkpoints, and metrics in Drive. A pilot confirms the commands work on bounded data; it is not evidence of a full corpus result. Keep the notebook output when reporting hardware, counts, class mix, and runtime.

In [ ]:
for label, path in {
    "input provenance": RUN_DIR / "input_provenance.json",
    "verifier split manifest": VERIFIER_DATA / "manifest.json",
    "FEVER holdout manifest": VERIFIER_HOLDOUT / "manifest.json",
    "verifier checkpoint": VERIFIER_CKPT / "config.json",
    "verifier metrics": VERIFIER_EVAL / "metrics.json",
    "repair split manifest": REPAIR_DATA / "repair_manifest.json",
    "repair checkpoint": REPAIR_CKPT / "config.json",
    "synthetic repair metrics": REPAIR_METRICS,
}.items():
    print(f"{label}: {'OK' if path.exists() else 'MISSING'} — {path}")
print("Pilot:", PILOT, "| Git commit:", commit, "| Drive run:", RUN_DIR)